# What the anchor does, beyond the perplexities

This notebook **continues [`two_domain_walkthrough.ipynb`](two_domain_walkthrough.ipynb)** and
needs the `lfa_demo/` workspace that notebook leaves behind. Run the walkthrough first, from the
same working directory; the setup cell below stops with a message rather than a `NameError` if it
cannot find it.

The walkthrough measured perplexity: what each stage learned and what it kept. Perplexity is the
right instrument for that — it is deterministic and it reads both axes at once — but it does not
say what the models *sound* like, and it does not separate the anchor from the dose. Three
questions here:

* **Is the control fair?** The walkthrough's two unanchored controls were trained at the
  *anchored* run's number of epochs, which is past their own optimum. Section 1 re-runs each at
  its own best dose, so the part of the gap that is dose rather than anchor is measured rather
  than waved at.
* **What do the three models actually say?** Ten fixed probes, identical decoding, every
  generation printed in full, including the ones that cut against the story.
* **Does the anchor change how the new domain and the model's prior knowledge reach each other?**
  Eight more probes, in both directions, each adjacent question paired with an unrelated one so
  that transfer can be told apart from the domain having eaten the model.

Everything generated here is **qualitative**. Nothing in this package scores generated text, no
claim rests on these panels, and a different set of questions could read differently. The
walkthrough's tables are where the evidence lives.

This notebook numbers its own three sections; where it points at the walkthrough it says so. The setup cell below is the one cell in this notebook that carries no
recorded output: it is new, and it re-establishes what the walkthrough left on disk — the demo
paths, the recipe, the fused stage-1 checkpoint, and the five-model table section 1 reads against. It re-scores those five models, which takes a few minutes on the card. Every cell after
it carries the output recorded on 2026-09-08, unaltered.


In [ ]:
# --- continue from the walkthrough --------------------------------------------------------------
# Nothing here is new work: it re-establishes the names `two_domain_walkthrough.ipynb` left on
# disk, in the same working directory. The five-model table is re-scored because section 1 reads
# its rows against the two dose-matched controls.
from pathlib import Path

DEMO = Path("lfa_demo").resolve()
if not (DEMO / "workspace" / "history.json").exists():
    raise SystemExit(
        f"no LFA workspace at {DEMO}. This notebook continues "
        f"examples/two_domain_walkthrough.ipynb -- run that one first, from this working "
        f"directory, and it will leave the lfa_demo/ this cell is looking for."
    )

import dataclasses, gc, json, logging, os, sys, time, textwrap

import torch
from peft import PeftModel

import lfa
from lfa import Recipe, Workspace
from lfa.corpus import load_corpus
from lfa.evaluate import domain_perplexity, wikitext2_perplexity
from lfa.models import load_teacher, load_tokenizer

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
logging.getLogger("datasets").setLevel(logging.WARNING)
logging.getLogger("lfa.evaluate").setLevel(logging.WARNING)

# The walkthrough's demo scale, unchanged -- these have to be the values that produced the runs
# on disk, not new ones.
EPOCHS = 4
N_WINDOWS = 200
DEVICE = "cuda:0"
ARTIFACT_ID = "qwen3-0.6b-gmm1543k-int8"
MODEL_ID = "Qwen/Qwen3-0.6B"

DATA, ARTIFACTS, WS = DEMO / "data", DEMO / "artifacts", DEMO / "workspace"
DARWIN_TRAIN, DARWIN_HELD = DATA / "darwin" / "train", DATA / "darwin" / "heldout"
COOKERY_TRAIN, COOKERY_HELD = DATA / "cookery" / "train", DATA / "cookery" / "heldout"

artifact_path = ARTIFACTS / f"{ARTIFACT_ID}.pt"
artifact_v2 = WS / "artifacts" / "v2.pt"
STAGE1_FUSED = WS / "models" / "stage1_fused"      # what stage 2 adapted
exported = WS / "models" / "stage2_fused_export"   # what the walkthrough's section 9 fused

recipe = Recipe.load("qwen3-0.6b")
tokenizer = load_tokenizer(MODEL_ID)


def held_out(path):
    corpus, _ = load_corpus(path, tokenizer, max_length=recipe.sequence_length,
                            val_fraction=0.0, seed=recipe.seed,
                            keep_short_whole=recipe.keep_short_whole)
    return corpus


DOMAINS = {"A · Darwin": held_out(DARWIN_HELD), "B · cookery": held_out(COOKERY_HELD)}

MODELS = {
    "base Qwen3-0.6B":            (MODEL_ID, None),
    "after A — anchored":         (MODEL_ID, WS / "runs" / "stage1" / "final_model"),
    "after A — NO anchor":        (MODEL_ID, WS / "runs" / "stage1_unanchored" / "final_model"),
    "after A+B — anchored":       (STAGE1_FUSED, WS / "runs" / "stage2" / "final_model"),
    "after A+B — NO anchor on B": (STAGE1_FUSED, WS / "runs" / "stage2_unanchored" / "final_model"),
}


def score(base, adapter):
    model = load_teacher(str(base), device=DEVICE, dtype=torch.bfloat16)
    if adapter is not None:
        model = PeftModel.from_pretrained(model, str(adapter))
    try:
        row = {"WikiText-2": wikitext2_perplexity(model, tokenizer, n_windows=N_WINDOWS,
                                                  device=DEVICE)}
        row.update({name: domain_perplexity(model, tokenizer, corpus, device=DEVICE)
                    for name, corpus in DOMAINS.items()})
        return row
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


table = {label: score(*spec) for label, spec in MODELS.items()}
base_row = table["base Qwen3-0.6B"]
print(f"lfa {lfa.__version__}; five models re-scored from {WS}")


## 1. Is the control fair? Both stages, at their own dose

There is a real objection to everything the walkthrough measured, and it is worth
measuring rather than waving at.

`EPOCHS = 4` was chosen from the **anchored** stage-1 run's held-out curve. Both unanchored
controls are trained at that same 4, because "the same run without the anchor" is what a control
is — but 4 is not *their* best dose. Without the anchor the model reaches the domain much faster
and then over-fits, and the trainer said so itself at the end of the stage-1 control ("this run
trained past its own optimum", lowest at epoch 2 and 12 % above it by epoch 4). The stage-2
control ends only 5 % above its own minimum — inside the 10 % the trainer treats as a wobble, so
it says nothing there — but its curve turns at epoch 3 all the same, and that is the dose the
cell below gives it. So part of every gap it printed is the anchor and part of it is three
extra epochs of unregularized training. Both curves are printed below, so the size of that
effect is visible rather than asserted.

The honest version of the question is: **at each run's own best dose, what is left?** Two more
runs answer it — each control again at the epoch *its own* curve bottomed at, read off that curve
in the cell below rather than written in, with its own complete learning-rate schedule, and each
in a workspace of its own so the walkthrough's chain is untouched.

In [18]:
import dataclasses
import json as _json


def curve(run_dir):
    return [(e["epoch"], e["val_perplexity"])
            for e in _json.loads((Path(run_dir) / "training_history.json").read_text())
            if "val_perplexity" in e]


print("held-out perplexity by epoch, on the trainer's own 10% split of each stage's training set")
for stage, runs in (("stage 1 (darwin) ", ("stage1", "stage1_unanchored")),
                    ("stage 2 (cookery)", ("stage2", "stage2_unanchored"))):
    print(f"  {stage}")
    for label, run in zip(("anchored ", "no anchor"), runs):
        print(f"     {label}  " + "  ".join(f"e{n}:{v:6.2f}" for n, v in curve(WS / "runs" / run)))

held-out perplexity by epoch, on the trainer's own 10% split of each stage's training set
  stage 1 (darwin) 
     anchored   e1: 18.52  e2: 17.38  e3: 16.91  e4: 17.05
     no anchor  e1: 16.57  e2: 14.80  e3: 15.53  e4: 16.62
  stage 2 (cookery)
     anchored   e1: 15.93  e2: 14.86  e3: 14.35  e4: 14.33
     no anchor  e1: 13.70  e2: 12.17  e3: 11.91  e4: 12.54


In [19]:
control = dataclasses.replace(recipe, lambda_qkv=0.0, lambda_mlp=0.0, mu=0.0)

# Each control's own best dose, read off ITS OWN curve above -- the rule that chose the anchored
# run's EPOCHS, applied to the run being asked about. Derived here rather than written in, so it
# cannot be inherited from an earlier run of this notebook.
DOSE_A = min(curve(WS / "runs" / "stage1_unanchored"), key=lambda point: point[1])[0]
DOSE_B = min(curve(WS / "runs" / "stage2_unanchored"), key=lambda point: point[1])[0]
print(f"each control's own best dose, from its own curve: stage 1 {DOSE_A} epochs, "
      f"stage 2 {DOSE_B} epochs (both anchored runs used {EPOCHS})")

t0 = time.time()

# Stage 1's control at its own dose: the base model, DOSE_A epochs, no anchor.
FAIR1 = DEMO / "workspace_dose_a"
fair1 = Workspace.init(FAIR1, MODEL_ID, artifact=str(artifact_path), artifact_id=ARTIFACT_ID)
fair1.train(DARWIN_TRAIN, recipe=control, epochs=DOSE_A, device=DEVICE)

# Stage 2's control at its own dose: the SAME starting model stage 2 had (the anchored stage-1
# fused checkpoint) and the same corpus, DOSE_B epochs, no anchor. A workspace of its own, so
# the walkthrough's chain keeps pointing at its real stage 2.
FAIR2 = DEMO / "workspace_dose_b"
fair2 = Workspace.init(FAIR2, str(STAGE1_FUSED), artifact=str(artifact_v2),
                       artifact_id=ARTIFACT_ID, recipe="qwen3-0.6b")
fair2.train(COOKERY_TRAIN, recipe=control, epochs=DOSE_B, device=DEVICE)

dose_seconds = time.time() - t0
print(f"\n{dose_seconds / 60:.1f} min")
print(f"  stage-1 control, {DOSE_A} epochs: " + "  ".join(f"e{n}:{v:6.2f}"
                                                          for n, v in curve(FAIR1 / "runs" / "stage1")))
print(f"  stage-2 control, {DOSE_B} epochs: " + "  ".join(f"e{n}:{v:6.2f}"
                                                          for n, v in curve(FAIR2 / "runs" / "stage1")))

each control's own best dose, from its own curve: stage 1 2 epochs, stage 2 3 epochs (both anchored runs used 4)
Local artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt recorded as the published 'qwen3-0.6b-gmm1543k-int8'


Artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt copied to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_a/artifacts/v1.pt


Default recipe for this workspace: 'qwen3-0.6b' (it names Qwen/Qwen3-0.6B)


Workspace initialised at /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_a over Qwen/Qwen3-0.6B


loader frame: short documents are kept whole in every epoch; a run under keep_short_whole=False sees them in fewer epochs, and perplexities are not comparable across the two


...

  Held-out: loss=2.4974 perplexity=12.151 over 10503 tokens


Final model saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_b/runs/stage1/final_model


  Training state saved (epoch=3, step=123, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_b/runs/stage1/training_history.json



3.5 min
  stage-1 control, 2 epochs: e1: 16.59  e2: 15.31
  stage-2 control, 3 epochs: e1: 13.73  e2: 12.03  e3: 12.15


In [20]:
dose_a = score(MODEL_ID, FAIR1 / "runs" / "stage1" / "final_model")
dose_b = score(STAGE1_FUSED, FAIR2 / "runs" / "stage1" / "final_model")


def block(title, rows, columns):
    width = max(len(label) for label, _ in rows)
    print(f"\n{title}")
    print(f"| {'':<{width}} | " + " | ".join(f"{c:>12}" for c in columns) + " |")
    print(f"| {'-' * width} | " + " | ".join("-" * 12 for _ in columns) + " |")
    for label, row in rows:
        print(f"| {label:<{width}} | " + " | ".join(f"{row[c]:12.2f}" for c in columns) + " |")


block("stage 1 — adapting the BASE model to Darwin",
      [("base model", base_row),
       (f"anchored, {EPOCHS} epochs", table["after A — anchored"]),
       (f"no anchor, {EPOCHS} epochs", table["after A — NO anchor"]),
       (f"no anchor, {DOSE_A} epochs", dose_a)],
      ["WikiText-2", "A · Darwin"])

block("stage 2 — adapting the same after-A model to the cookbook",
      [("model it started from", table["after A — anchored"]),
       (f"anchored, {EPOCHS} epochs", table["after A+B — anchored"]),
       (f"no anchor, {EPOCHS} epochs", table["after A+B — NO anchor on B"]),
       (f"no anchor, {DOSE_B} epochs", dose_b)],
      ["WikiText-2", "A · Darwin", "B · cookery"])

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).



stage 1 — adapting the BASE model to Darwin
|                     |   WikiText-2 |   A · Darwin |
| ------------------- | ------------ | ------------ |
| base model          |        17.80 |        30.12 |
| anchored, 4 epochs  |        16.09 |        17.45 |
| no anchor, 4 epochs |        24.92 |        17.10 |
| no anchor, 2 epochs |        18.87 |        15.70 |

stage 2 — adapting the same after-A model to the cookbook
|                       |   WikiText-2 |   A · Darwin |  B · cookery |
| --------------------- | ------------ | ------------ | ------------ |
| model it started from |        16.09 |        17.45 |        23.02 |
| anchored, 4 epochs    |        15.94 |        18.92 |        15.13 |
| no anchor, 4 epochs   |        22.59 |        31.45 |        13.37 |
| no anchor, 3 epochs   |        19.75 |        27.02 |        12.99 |


Read those on **all** the columns at once, which is the whole discipline of the method. The
question is never "which run has the better WikiText-2"; it is "at comparable fit on the domain
being learned, which run kept more of everything else". A control that is worse everywhere has
been beaten outright. A control that is *better* on the new domain and worse elsewhere has
bought the domain by spending what it already had — which is the trade the anchor exists to
change, not a trade it claims to abolish.

Two things this section is honest about, whichever way the numbers fell:

* Some of the gap in the walkthrough's sections 5 and 8 is dose, not anchor. Reporting
  only the 4-epoch controls would have overstated the anchor by the difference between
  the two control rows here.
* A 2-epoch run is not simply a truncated 4-epoch run: the learning-rate schedule is laid over
  whatever number of epochs you ask for, so these are complete runs with their own warmup and
  decay. That is the right comparison, and it is also a *second* difference from the 4-epoch
  control, not a clean single-variable change.

And, still: one seed, one run per row, 193 documents.

## 2. What the three models actually say

Everything above is a perplexity. Perplexity is the right instrument for this method — it is
deterministic, it reads all three axes at once, and the walkthrough's sections 5 and 8, and section 1 above, are argued on it. What it does not tell you is what the models *sound*
like, and that is worth seeing too.

So: five probes per stage, put to three models each under identical settings — **greedy decoding,
200 new tokens, `repetition_penalty=1.15`, `no_repeat_ngram_size=6`**, the chat template with
Qwen3's thinking mode off. Two of the ten slots are deliberate repeats (`A1` asks about domain A
again after domain B has been learned; `G1` is the same general question in both panels), which
makes thirty generations from eight questions. The probe set was written down before any output
was looked at, and nothing in it was reworded, dropped or swapped afterwards. Every generation is
printed in full, including the ones that cut against the story.

**Why those two repetition controls.** Plain greedy decoding with no repetition penalty makes a
0.6B model repeat a clause until the token budget runs out, and it did: an earlier decode of these
same ten probes had every trained arm looping, one of them five times over. That is a property of
the decoder, not of any model, and it makes the panel unreadable. The penalty (the standard mild
band) and the 6-token no-repeat window are set **identically for every model**, including where
they cost one — they were chosen on whether they clear the loops, not on whose answers they
improve, and the base model pays for them too (see `G2`). Decoding stays deterministic, so there
is no seed to choose and no run to repeat until it looks good.

Three families, so it is clear what to look at:

| | |
|---|---|
| `A*`, `B*` | **adaptation** — does the model know the domain it was just trained on |
| `G*` | **retained general capability** — the axis forgetting destroys first |
| `I*` | **integration** — domain content delivered inside behaviour the base model already had: register control, brevity, doing what was asked |

**This is qualitative illustration, not evidence.** One greedy sample per model per prompt is not
a population, nothing here is scored by anything, and no claim in this notebook rests on it. The
measurements are the tables above.

One thing the second panel needs said plainly: **both** of its trained models start from the
*anchored* stage-1 checkpoint. It isolates stage-2 anchoring on its own; it is not an anchored
chain against an unanchored one.

In [24]:
# The probe set is fixed before any output is seen, and nothing below is dropped or shortened
# afterwards. `A*`/`B*` ask whether the domain was learned; `G*` ask whether general capability
# survived, which is the axis forgetting destroys first; `I*` ask for domain content delivered
# inside behaviour the base model already had -- register, brevity, following the instruction.
DARWIN_Q = "What did Darwin mean by natural selection?"
GENERAL_Q = "What is photosynthesis? Answer in two sentences."

PANEL_A = [
    ("A1", DARWIN_Q),
    ("A2", "According to The Origin of Species, what part does the struggle for existence play "
           "in the modification of species?"),
    ("G1", GENERAL_Q),
    ("G2", "List three prime numbers greater than 10."),
    ("I1", "In two sentences, explain to a ten-year-old why some animals in a species survive "
           "and others do not."),
]

PANEL_B = [
    ("A1", DARWIN_Q),                                    # the retention probe, repeated
    ("B1", "How do you make a plain suet pudding?"),
    ("B2", "What is the proper way to roast a joint of beef?"),
    ("G1", GENERAL_Q),                                   # repeated
    ("I2", "Give a short recipe for a simple soup, then explain in one sentence why boiling "
           "changes the food."),
]


# Decoding, identical for every model and every probe. Still greedy, so the section is
# deterministic and reproduces on a re-run; the two repetition controls are there because plain
# greedy decoding makes a 0.6B model repeat a clause until the budget runs out, which is a
# property of the decoder and would be easy to misread as a property of a model.
GENERATION = dict(
    max_new_tokens=200,        # unchanged -- room for a numbered recipe or a short explanation
    do_sample=False,           # greedy: no seed, no run-to-run variation, nothing to re-roll
    repetition_penalty=1.15,   # the standard mild band; 1.1 left some answers still looping
    no_repeat_ngram_size=6,    # hard block on repeating a 6-token span -- short domain phrases
                               # such as "struggle for existence" still come through
)


def answer(model, question):
    """One deterministic answer, chat template, thinking off -- GENERATION for every model."""
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": question}],
                                           tokenize=False, add_generation_prompt=True,
                                           enable_thinking=False)
    ids = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**ids, pad_token_id=tokenizer.eos_token_id, **GENERATION)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def run_panel(models, probes):
    """Every model's answer to every probe. Loads one model at a time; nothing is filtered."""
    answers = {}
    for label, (base, adapter) in models.items():
        model = load_teacher(str(base), device=DEVICE, dtype=torch.bfloat16)
        if adapter is not None:
            model = PeftModel.from_pretrained(model, str(adapter))
        try:
            answers[label] = {tag: answer(model, question) for tag, question in probes}
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
    return answers


def show(answers, probes):
    for tag, question in probes:
        print(f"\n{'=' * 96}\n{tag}  {question}\n{'=' * 96}")
        for label, by_tag in answers.items():
            body = textwrap.fill(by_tag[tag], 92, initial_indent="    ", subsequent_indent="    ")
            print(f"\n  [{label}]\n{body}")


STAGE1_PANEL = {
    "base":       (MODEL_ID, None),
    "unanchored": (MODEL_ID, FAIR1 / "runs" / "stage1" / "final_model"),
    "anchored":   (STAGE1_FUSED, None),
}

t0 = time.time()
answers_a = run_panel(STAGE1_PANEL, PANEL_A)
print(f"stage-1 panel: 3 models x 5 probes, {time.time() - t0:.0f} s\n"
      f"  base        Qwen3-0.6B, untouched\n"
      f"  unanchored  domain A with lambda = mu = 0, at its own best dose ({DOSE_A} epochs)\n"
      f"  anchored    domain A with the recipe's anchor ({EPOCHS} epochs)")
show(answers_a, PANEL_A)

stage-1 panel: 3 models x 5 probes, 54 s
  base        Qwen3-0.6B, untouched
  unanchored  domain A with lambda = mu = 0, at its own best dose (2 epochs)
  anchored    domain A with the recipe's anchor (4 epochs)

A1  What did Darwin mean by natural selection?

  [base]
    Charles Darwin's concept of **natural selection** is a fundamental principle in
    evolutionary biology, and it has had a profound impact on the understanding of life on
    Earth. Here’s what he meant:  ### 1. **The Origin of Species**:      Darwin proposed
    that species evolve over time through **natural selection**, where organisms with traits
    better suited to their environment are more likely to survive and reproduce. This leads
    to the gradual change in populations.  ### 2. **Survival of the Fittest**:      He
    emphasized that individuals who possess advantageous traits (e.g., physical strength,
    intelligence) have higher chances of survival and reproduction, which in turn promotes
    the spre

In [25]:
# Both stage-2 arms start from the SAME model: the anchored stage-1 checkpoint. So this panel
# isolates stage-2 anchoring alone -- it is not an anchored chain against an unanchored one.
STAGE2_PANEL = {
    "after A":    (STAGE1_FUSED, None),
    "unanchored": (STAGE1_FUSED, FAIR2 / "runs" / "stage1" / "final_model"),
    "anchored":   (exported, None),
}

t0 = time.time()
answers_b = run_panel(STAGE2_PANEL, PANEL_B)
print(f"stage-2 panel: 3 models x 5 probes, {time.time() - t0:.0f} s\n"
      f"  after A     the anchored stage-1 model, before the cookbook\n"
      f"  unanchored  domain B on top of it with lambda = mu = 0, own best dose ({DOSE_B} epochs)\n"
      f"  anchored    domain B on top of it with the anchor ({EPOCHS} epochs)")
show(answers_b, PANEL_B)

stage-2 panel: 3 models x 5 probes, 51 s
  after A     the anchored stage-1 model, before the cookbook
  unanchored  domain B on top of it with lambda = mu = 0, own best dose (3 epochs)
  anchored    domain B on top of it with the anchor (4 epochs)

A1  What did Darwin mean by natural selection?

  [after A]
    Darwin's theory of natural selection is a fundamental principle in evolutionary biology.
    He explained that all living beings have been created through the gradual process of
    modification and inheritance, which he called "natural selection." Here are some key
    points:  1. **Gradual Change**: Natural selection acts over long periods (millions to
    millions of years) on inherited variations. 2. **Inheritance of Variations**: These
    variations become more common or advantageous as they persist in successive generations.
    3. **Direction of Change**: The changes tend to be directed toward improving existing
    forms. 4. **Natural Selection Produces New Species**: 

### What the probes showed, including what they did not

**The decoder first, because it changes how the rest reads.** An earlier decode of these same ten
probes, at the same 200 greedy tokens but with no repetition controls, had *every* trained arm
looping: the stage-2 anchored model answered `B1` by repeating "Put a pound of butter in a large
cask, and pour in the suet pudding, and stir it well" five times and starting a sixth. None of
that survives the settings above. Across all thirty generations exactly one contains a repeated
span of five words or longer, and it is a topic phrase — "the inhabitants of an island", three
times inside one long answer — not a loop. **The degenerate output was the decoder.** Everything
below is what the models do once it is taken away.

**What the controls cost, stated before it is used.** `G2` is the receipt. Under plain greedy
decoding all three stage-1 models opened with "Three prime numbers greater than 10 are"; with the
repetition penalty on, the base model says "greater than **2**" and both trained models say
"greater than **9**". The penalty down-weights a token the prompt already used, and the base model
pays for it exactly as the others do. All three still list three correct primes above 10. This is
the price of a readable panel, and it is charged to every model equally.

**Adaptation is visible, but not on both arms equally.** The unanchored stage-1 control's
adaptation is total and unmistakable: it answers all five probes — including both general ones —
in Darwin's own first person and in the book's argumentative style. The anchored model's is much
harder to see in the text. It uses the domain's vocabulary on `A1` and `A2` ("struggle for
existence", "variations", "modification") but keeps the base model's register: numbered points and
a closing summary. Shown only its stage-1 answers, you would struggle to say what corpus it had
been trained on. That is a real limitation of this panel, and it points at the table: section 1 puts the anchored run at 17.45 on Darwin against the base model's 30.12, so the adaptation is
there — it is just not what the prose style shows you.

**On the general axis the control's damage is now legible, and it was not before.** This is the
biggest change from the looping decode, where `G1` and `G2` looked like ties and the section had to
say that nothing catastrophic had happened and no more. Asked what photosynthesis is, the stage-1
control opens correctly and then drifts into invented Victorian biology — "the greater the amount
of oxygen produced, the more rapidly the cells will divide and become larger" — for the rest of its
budget. Asked on `I1` for two sentences for a ten-year-old, it produces a passage about island
faunas containing a fabricated statistic ("about 1000 fish in the sea, compared with only 250 in
the open ocean over the whole world"). The anchored model answers `I1` in exactly two plain
sentences, using natural selection as the explanation — domain content delivered inside behaviour
the base model already had, which is what `I*` was written to look for. Section 1's WikiText-2
column had said the same thing first: 18.87 for that control against 16.09 anchored, base 17.80.

**Stage 2 splits, and both halves agree with the table.** Retention went the way the method
predicts and by a wide margin. Asked about natural selection after learning the cookbook, the
unanchored arm answers about Darwin's *biography* and invents a quotation for him — "I have known
many persons who were born with disabilities, but succeeded in their own ways" — while the anchored
arm stays on the concept ("he said that these traits were not accidental but had been selected from
other forms of life"). A · Darwin: 27.02 for the control, 18.92 anchored.

The new domain went the other way, and just as plainly. On `B1` the control writes the fullest
period recipe — "Take three quarts of milk, two pounds and a half of sugar, one pound of butter,
and the same of lard; put them in a dutch-oven" — while the anchored model's is in the same voice
but vaguer and more confused, opening "Put together the ingredients, and put them in a large pot"
and later calling for "stirring-pieces of nutmegs". On `B2` the control gives a real 1853 method;
the anchored model, asked how to **roast**, instructs you to boil enough water to cover the meat
and roast it in that. Both answers are coherent — that part was the decoder — but on the domain it
just learned the control is better, which is what B · cookery 12.99 against 15.13 said first. **The
anchor makes that trade; it does not abolish it.**

The remaining probes are ties. `G1` in the second panel: all three give two correct sentences.
`I2`: both trained models give a recipe and the one-sentence explanation asked for, and the only
model that fails is the one that has *not* seen domain B — it announces "**A simple soup recipe:**"
and then describes boiling rather than giving a recipe. `A2` is a tie of two different kinds: the
control argues the book's case in the book's voice, the anchored model answers the question in the
base model's voice with the domain's terms, and nothing here can rank those.

**The count.** By the narrow criterion of answering the question asked, in the form asked, without
inventing: the anchored model is better on four probes (`A1`, `G1` and `I1` in the first panel,
`A1` in the second), worse on two (`B1`, `B2`), and indistinguishable on the remaining four. It
does not adapt *better* on either domain at either stage; on both it fits the corpus less well than
the control does, and the tables said so first.

**Where the same model appears in both panels its answers are identical** — stage 1's `anchored`
and stage 2's `after A` are the same checkpoint, and their `A1` and `G1` answers match character
for character. Decoding is deterministic, so re-running this section reproduces it.

**And the standing caveat, which the cleaner output does not weaken.** Thirty single greedy
generations from one 0.6B model are an illustration, not a measurement. Nothing here is scored by
anything, no claim in this notebook rests on it, and a different set of ten questions could tell a
different story. The perplexity tables are the evidence; this is what the evidence sounds like.


## 3. Integration, and what it should do in both directions

Everything up to here measures how much the model **keeps**. That framing makes the anchor sound
like a brake — something that stops training before it goes too far — and a brake is not very
interesting. The mechanism says something stronger than "less is lost", and this section states
what it says and then puts it to the models to see how much of it actually shows at this scale.

### What the anchor does, mechanically

One term does the work. For each sub-module of the network — an attention projection, an MLP
block — the training loss carries

> `λ · E` over `h ~ p(h)` of `‖ f_student(h) − f_teacher(h) ‖²`

Read it right to left. `f` is one sub-module, treated as a function. `h` is a *hidden state* — one
of the vectors that flows between the model's layers, which is what a sub-module actually takes as
input. `p(h)` is a distribution over those vectors, estimated from the **base** model on general
text and shipped inside the artifact the walkthrough fetched; sampling from it costs no data and no
forward pass over anything the model once saw. So the term draws inputs the base model already
produces, pushes each through one sub-module in both the old model and the one being trained, and
charges for the difference in what comes out.

What that penalises is not weight movement as such — the much smaller `μ` term does that. It
penalises **changing what a piece of the model computes on the inputs it already sees**. A weight
change that leaves every familiar hidden state mapped where it was is free no matter how large it
is; one that re-purposes a sub-module is expensive no matter how small.

That leaves gradient descent an easy route to the new corpus and a hard one. The hard route is to
overwrite — carve out fresh machinery for the cookbook and pay for every familiar input that
machinery no longer handles the way it did. The easy route is to reach the same behaviour
*through* structure that already exists: encode "brown the meat first" using whatever the model
already has for heat, for food, for cause and effect. The anchor cannot tell the two routes apart
and does not try. It just makes the first one expensive. That is the whole of the constraint, and
the phrase for adapting under it is **guided integration**.

### Why a constraint like that predicts traffic in both directions

If new content has to be written through machinery that already exists, then the new content and
the old are not in separate compartments — they are expressed in the same representations, so each
should be reachable from the other. Two predictions follow, and they are not the same prediction:

- **Inward — prior knowledge reaching the new domain.** Ask, about the corpus's own subject,
  something the corpus does not contain. Every corpus is a slice of a subject, never the whole of
  it: *The Origin of Species* (1859) predates genetics and germ theory, and an 1853 cookbook gives
  method without chemistry. A model that has integrated the corpus should still be able to bring
  what it already knew to those questions.
- **Outward — domain semantics reaching adjacent questions.** Ask something the corpus is *not*
  about but which shares its concepts. The corpus should show up in the answer, because that is
  where the new content was written.

### Outward transfer is only a virtue if it is selective

This is the part that needs the control, and section 2 has already shown why. The unanchored
stage-1 model transfers the domain **everywhere**: asked what photosynthesis is, it answers
correctly for one sentence and then drifts into invented Victorian biology; asked for two sentences
for a ten-year-old, it returns a passage about island faunas with a fabricated statistic. Domain
semantics arriving in a question that has nothing to do with the domain is not integration — it is
the domain having eaten the model, and it is the ordinary failure mode of fine-tuning a small model
on a small corpus.

So transfer on its own proves nothing, and a panel that only asked adjacent questions would be
unreadable. Each adjacent question below is therefore paired with an **unrelated** one, and the
demonstration is the *difference between the two*, not either alone. A model that draws on the
corpus for the adjacent question and leaves it out of the unrelated one has done something the
model that does it for both has not.

### The standing of this claim, stated before the outputs

The mechanism above is arithmetic: it is what the loss term computes, and that is not in question.
Whether the resulting model is *better integrated* in any measurable sense is a different question
with a worse answer. The research programme behind this package built two structural measures of
integration and neither predicted judged quality — one of them came out with the sign backwards.
So: what follows is eight questions put to six models, every generation printed in full and
uncurated. It illustrates behaviour consistent with the mechanism. It is not evidence for it,
nothing in this notebook rests on it, and a different eight questions could read differently.

| | |
|---|---|
| `P*` | **inward** — about the domain's subject, answerable only from knowledge the corpus does not hold |
| `Q*` | **outward, adjacent** — not about the corpus, but sharing its concepts; domain content here *fits* |
| `N*` | **outward control, unrelated** — shares nothing with either corpus; domain content here is a **fault** |

The eight probes were written down before any output was looked at, and none was reworded, dropped
or reordered afterwards. `Q3` is the one that reads both stages at once: it is adjacent to domain
A, and it is put to models that have since been trained on domain B. `N1` and `N2` are the same
question, so the two panels can be read against each other. Decoding is exactly section 2's —
greedy, 200 new tokens, `repetition_penalty=1.15`, `no_repeat_ngram_size=6`, the chat template with
thinking off — identical for every model and every probe, with nothing tuned per question.

In [ ]:
# The transfer probes, fixed before any output was seen: nothing below was reworded, dropped or
# reordered afterwards. Same three models as section 2, the same `GENERATION` settings and the
# same helpers -- only the questions are new.
#
#   P*  inward   -- knowledge the corpus does NOT contain, asked about the corpus's own subject
#   Q*  outward  -- a question the corpus is not about, but which shares semantics with it
#   N*  control  -- a question sharing nothing with either corpus, where transfer is a FAULT
PANEL_T1 = [
    ("P1", "How does natural selection relate to genes and DNA?"),
    ("P2", "Does natural selection explain why bacteria become resistant to antibiotics?"),
    ("Q1", "Why do species introduced to a new island often multiply rapidly?"),
    ("N1", "What is the capital of Japan, and roughly how many people live there?"),
]

t0 = time.time()
answers_t1 = run_panel(STAGE1_PANEL, PANEL_T1)
print(f"stage-1 transfer panel: 3 models x 4 probes, {time.time() - t0:.0f} s\n"
      f"  P1 P2 inward   -- 1859 predates genetics and germ theory; the corpus cannot supply either\n"
      f"  Q1    outward  -- island colonisation is adjacent to the corpus, not in the question\n"
      f"  N1    control  -- unrelated; domain vocabulary appearing here is the failure mode")
show(answers_t1, PANEL_T1)

stage-1 transfer panel: 3 models x 4 probes, 59 s
  P1 P2 inward   -- 1859 predates genetics and germ theory; the corpus cannot supply either
  Q1    outward  -- island colonisation is adjacent to the corpus, not in the question
  N1    control  -- unrelated; domain vocabulary appearing here is the failure mode

P1  How does natural selection relate to genes and DNA?

  [base]
    Natural selection is a fundamental mechanism of evolution, but it operates through
    **genetic variation**, not directly through **genes** or **DNA itself**. Here's how they
    relate:  ### 1. **Genes are the basis of heredity**    - Genes (DNA sequences) carry
    instructions for building proteins, which in turn determine traits.    - These genetic
    information is passed from parents to offspring.  ### 2. **Variation occurs due to
    mutations**    - Mutations can introduce new alleles (different versions of a gene),
    leading to genetic diversity.    - This variability is crucial because it allows

In [ ]:
# Stage 2, same three models as the stage-2 panel above -- all starting from the anchored
# stage-1 checkpoint. `Q3` is the one that reads both stages at once: it is adjacent to domain A,
# put to models that have since been trained on domain B.
PANEL_T2 = [
    ("P3", "Why does browning meat before stewing improve the flavour?"),
    ("Q2", "How did households keep food from spoiling before refrigeration?"),
    ("N2", "What is the capital of Japan, and roughly how many people live there?"),
    ("Q3", "In a place with little food, what happens to a population of animals over many "
           "generations?"),
]

t0 = time.time()
answers_t2 = run_panel(STAGE2_PANEL, PANEL_T2)
print(f"stage-2 transfer panel: 3 models x 4 probes, {time.time() - t0:.0f} s\n"
      f"  P3 inward   -- an 1853 cookbook gives method, not the chemistry of why it works\n"
      f"  Q2 outward  -- adjacent to domain B\n"
      f"  N2 control  -- the same unrelated question as N1, so the two panels can be compared\n"
      f"  Q3 outward  -- adjacent to domain A, asked after domain B was learned")
show(answers_t2, PANEL_T2)

stage-2 transfer panel: 3 models x 4 probes, 55 s
  P3 inward   -- an 1853 cookbook gives method, not the chemistry of why it works
  Q2 outward  -- adjacent to domain B
  N2 control  -- the same unrelated question as N1, so the two panels can be compared
  Q3 outward  -- adjacent to domain A, asked after domain B was learned

P3  Why does browning meat before stewing improve the flavour?

  [after A]
    The improvement of flavor in meat when browned before stewing is due to several factors:
    1. **Increased Fat and Moisture**: Brown meat contains more fat, moisture, and protein
    than white meat. These substances are better absorbed by water-soluble ingredients like
    starches or oils during cooking.  2. **Improved Texture**: The brown color suggests a
    well-cooked surface with less dryness, which enhances the texture and taste.  3.
    **Enhanced Flavor Profile**: When cooked brownly, the meat has a richer flavor because
    it contains more aromatic compounds (like tannins

### What the panel showed, and which half of the idea did not appear

**Outward transfer is visible, and — the part that matters — it is selective.** Four exhibits;
the first two are a pair and have to be read together.

`N1` is the clearest of the four. The question has nothing to do with Darwin. The unanchored
control answers it correctly in two sentences and then spends its remaining 165 words on Victorian
biogeography: *"The capital of Japan is Tokyo. It has a population of about 10 million
inhabitants. **In South America, we find that the number of inhabitants in each country varies
greatly from one to another**…"*, on through the southern United States, Australia and New Zealand.
The anchored model answers in four sentences about Tokyo's population and its role in trade and
culture, and stops. Both models learned the same corpus; only one of them brings it to a geography question.

`Q1` is the other half of that pair, and it has to be read against `N1` to mean anything. Asked why
species introduced to an island multiply, the **base** model gives six generic factors and never
names natural selection at all — its list runs "Adaptation and Survival", "Genetic Diversity",
"Resource Availability", "Human Intervention". The anchored model leads with it: *"1. **Natural
Selection**: The introduction of new species can lead to natural selection, as the best-adapted
individuals will survive and reproduce more successfully"*, then competition for resources and
altered niches. The domain's central concept arrives on an adjacent question where the base model
did not raise it — and does not arrive on `N1`. The unanchored control also leads with natural
selection on `Q1`; it just does the same thing on `N1`. **The difference between the two arms is
not whether the domain transfers, it is whether the transfer is aimed.**

`Q3` carries the most weight of the four, because it reads both stages at once: a
domain-A-adjacent question put to models that have since been trained on the cookbook. The anchored arm still has
Darwin — *"the population would still diminish by natural selection: for instance, when a new
variety of cattle was introduced into an area which had no such cattle, the old stock would
gradually give way to this new one"*. The unanchored arm has lost the subject completely and
answers in the cookbook's household register: *"A small group of animals in an unimproved state
will soon become sick and weak… It is very important to keep children well fed"*. The string
"natural selection" does not occur in its answer. (One caution on the anchored answer: cattle,
sheep and flocks belong to *both* corpora — Darwin opens on domestic breeds and the cookbook is
full of meat — so the livestock framing cannot be attributed to either.)

`Q2` is a quieter version of the same thing on domain B. Asked how food was kept before
refrigeration, the model that has **not** seen the cookbook invents a history of canning, complete
with *"Some culls (like those made by bees)"* and a claim that canning declined after the 18th
century. The anchored model answers from the period's actual practice — a wooden box or tin case,
*"you can use salt as an ingredient"*, *"a little vinegar in the box"* — which is domain content
arriving where it belongs. It is a mixed answer, with two garbled sentences at the end, and it is
recorded above in full.

**Inward transfer shows at stage 1, and stage 2 cannot test it at this scale.** On `P1` and `P2`
the base model knows genetics and antibiotic resistance well, so there is something to transfer,
and the two trained arms separate sharply. The unanchored model has simply lost the knowledge:
its `P1` answer never contains the string "DNA" at all, opening *"Genes are the basic units of
variation, and they act through inheritance"* before dissolving into *The Origin*'s prose about
varieties in close contact. The anchored model keeps it and frames it in the domain's terms —
*"DNA provides the blueprint, but natural selection determines which traits become dominant"* —
and on `P2` gives horizontal gene transfer and selection pressure, neither of which any 1859 text
can supply.

`P3` fails to test anything, and that is worth saying plainly rather than reading a result into
it. Asked why browning meat improves flavour, **none** of the three models knows the chemistry —
not even "after A", which has never seen a cookbook and is the closest thing here to an untouched
control. It answers that *"Brown meat contains more fat, moisture, and protein than white meat"*,
which confuses browning with brown meat and is wrong. Prior knowledge cannot transfer inward if
the model never had it, and a 0.6B model does not have the Maillard reaction. The probe measures
the model's size, not the method.

**What went against the anchored arm, uncurated.** `N2` is the anchored model's worst answer in the
section: it gives Tokyo correctly in two sentences, then invents a question of its own —
*"What are some natural wonders in Europe?"* — and lists thirty of them, including the Welsh Wastes,
the Spanish Andes and the Polish Poles. By the reading rule set out above it still *passes* `N2`:
no cookery vocabulary appears, so nothing transferred where it should not. But it is a bad answer,
the unanchored control's thirteen words are the best one on that probe, and the selectivity result
at stage 2 rests on the absence of a fault rather than on quality. Two smaller ones: the **base**
model answers `N1` with "Osaka City", so the control question cannot be read as the trained models
degrading — at this size the base model is not a gold standard either. And the anchored `P1` answer
opens with a claim that is arguably wrong ("operates on the variation of individuals, not on their
genes or DNA") where the base model is more careful.

**The summary, direction by direction.** Outward transfer and its selectivity: **visible**, at both
stages, most convincingly on `N1`/`Q1` as a pair and on `Q3`. Inward transfer: **visible at stage
1**, on `P1` and `P2`, where it takes the form of the anchored model retaining knowledge the
control has lost and answering with it inside the domain's framing; **not testable at stage 2**,
because the prior knowledge `P3` asks for is not in a model this small. And the standing caveat is
unchanged by any of it — twenty-four greedy generations are an illustration of a mechanism, not a
measurement of one, and the tables above are still where the evidence in this notebook lives.

## Where that leaves the pair of notebooks

Section 1 split the walkthrough's gaps into dose and anchor by giving each control its own best
number of epochs. Sections 2 and 3 said what the surviving difference looks like in the models'
own words: an unanchored model that answers a question about the capital of Japan and then slides
into Victorian biogeography for another 165 words, beside an anchored one that answers in four
sentences and stops — while both bring natural selection to a question about island species that
the untrained model answers without ever naming it. Transfer where it fits, and not where it does
not.

And the caveat that has stood over every panel here: one seed, one run per row, single greedy
generations from one 0.6B model. These are illustrations of a mechanism, not measurements of one.
The walkthrough's tables are the evidence; this is what the evidence sounds like.

Back to [`two_domain_walkthrough.ipynb`](two_domain_walkthrough.ipynb) for the flow itself, or to
`docs/concepts.md` for what the anchor does and how a run is read.
